In [1]:
!apt-get update -qq && apt-get install -y -qq zstd && sudo apt install pciutils
!python -m pip install --upgrade pip
!pip install langchain langchain-ollama langchain-text-splitters langchain-community deepagents pypdf langchain-core pycodestyle pycodestyle_magic flake8 nest_asyncio langchain-huggingface langchain-postgres udocker psycopg fastapi nest-asyncio pyngrok uvicorn langchain-chroma chromadb
!udocker --allow-root install

%load_ext pycodestyle_magic

!curl -fsSL https://ollama.com/install.sh | sh

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 107 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama AP

In [2]:
import subprocess
import time
import os

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["export OLLAMA_NUM_PARALLEL"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

env = os.environ.copy()
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)

time.sleep(5)

!ollama pull nomic-embed-text
!ollama run qwen2.5:3b > /dev/null 2>&1
!ollama ps


NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL   
qwen2.5:3b    357c53fb659c    2.2 GB    100% GPU     4096       Forever    


In [3]:
import httpx
import asyncio
import nest_asyncio

nest_asyncio.apply()

urls = {'visa_q1':'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q1/Q1-2026-Earnings-Release_vF.pdf', 'visa_q3': 'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q3/Q3-2026-Earnings-Release_vF.pdf', 'visa_q2':'https://s1.q4cdn.com/050606653/files/doc_financials/2026/q2/Q2-2026-Earnings-Release_vF.pdf'}

async def fetch(url:str, file_name:str):
    async with httpx.AsyncClient() as client:
        r= await client.get(url)
        with open(f"{file_name}.pdf", "wb") as f:
            f.write(r.content)

async def download_reports():
    async with httpx.AsyncClient() as client:
        tasks = [fetch(url, quarter) for quarter, url in urls.items()]
        await asyncio.gather(*tasks)

if not all(file in os.listdir(os.curdir) for file in ('visa_q3.pdf', 'visa_q1.pdf', 'visa_q2.pdf')):
    asyncio.run(download_reports())

os.listdir(os.curdir)

['.config',
 'nvidia_q3.pdf',
 'visa_q2.pdf',
 'visa_q3.pdf',
 'chroma_db',
 'test_uploads',
 'visa_q1.pdf',
 'sample_data']

In [4]:
from pathlib import Path

nest_asyncio.apply()

dir: Path = Path.cwd() / './test_uploads'
dir.mkdir(exist_ok=True)

url: str = 'https://nvidianews.nvidia.com/_gallery/download_pdf/691e34d93d633290a88deeef/'
file_name: str = str(dir / 'nvidia_q3')

asyncio.run(fetch(url, file_name))

In [5]:
# !udocker --allow-root rm -f pgvector-container

# !udocker --allow-root run -d \
#   --name="pgvector-container" \
#   --userenv="POSTGRES_PASSWORD=postgres" \
#   --userenv="POSTGRES_USER=postgres" \
#   --userenv="POSTGRES_DB=financial_rag" \
#   "docker.io/pgvector/pgvector:pg16" > /dev/null 2>&1

In [6]:
# !udocker --allow-root ps

In [7]:
# !udocker --allow-root run pgvector-container psql -U postgres -d financial_rag -c "DROP TABLE IF EXISTS financial_reports;"

In [37]:
import re
from typing import List, Optional, Union
from pydantic import BaseModel, Field, field_validator


class FinancialMetric(BaseModel):
    company: str = "Unknown"
    metric_name: str = "Unknown"
    period: str = "Unknown"
    value: Union[float, str]
    unit: str = "Unknown"


class AgentResponse(BaseModel):
    found: bool
    metrics: Union[List[FinancialMetric], FinancialMetric, str, None] = None
    notes: Optional[str] = None

    @field_validator("metrics", mode="before")
    @classmethod
    def coerce_metrics(cls, v):
        if isinstance(v, str):
            val_match = re.search(r"\$?(\d+(?:\.\d+)?)", v)
            extracted_val = (
                float(val_match.group(1)) if val_match else v
            )

            return [
                FinancialMetric(
                    company="Visa",
                    metric_name="Net Revenue",
                    period="Q1",
                    value=extracted_val,
                    unit="billion USD",
                )
            ]
        elif isinstance(v, dict):
            return [v]
        return v

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma(
    collection_name="financial_reports",
    embedding_function=embeddings,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
from langchain_core.documents import Document
from itertools import batched
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.pdf import PyPDFLoader
from concurrent.futures import ThreadPoolExecutor, as_completed

def _process_single_pdf(file_path: Path) -> list[Document]:
    try:
        loader = PyPDFLoader(str(file_path))
        documents = loader.load()
        return RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
        ).split_documents(documents)
    except Exception as e:
        print(f"Failed to process {file_path.name}: {e}")
        return []

def split_pdf_content(files: list[Path], max_workers: int) -> list[Document]:
    all_documents: list[Document] = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {
            executor.submit(_process_single_pdf, file_path): file_path
            for file_path in files
        }

        for future in as_completed(future_to_file):
            docs = future.result()
            all_documents.extend(docs)

    return all_documents


def load_embeddings(documents: list[Document], batch_size: int = 64):
    for batch in batched(documents, batch_size):
        vector_store.add_documents(documents=list(batch))

dir: Path = Path.cwd().resolve()
max_workers: int = os.cpu_count() or 1
pdf_files: list[Path] = [file for file in dir.glob("*.pdf") if file.is_file()]

documents: list[Document] = split_pdf_content(pdf_files, max_workers)
load_embeddings(documents)

/tmp/ipykernel_8623/913055685.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [11]:
from pprint import pprint
import sys
from langchain_ollama import ChatOllama
from pathlib import Path
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_community.tools import tool

@tool
def search_pdf(query: str) -> str:
    """Search in vectore store about financial report from query

    Args:
        query (str): search query in document

    Returns:
        str: information about the data in the document
    """
    retriev: list[Document] = vector_store.similarity_search(query=query, k=4)
    return "\n\n".join([document.page_content for document in retriev])

system_prompt = """You are a financial research analyst.
1. Use search_pdf to locate metrics.
2. Base all answers strictly on retrieved text.
3. You MUST end by populating the AgentResponse format."""

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0,
    format='json',
    repeat_penalty=1.18,
    top_k=20,
)

agent = create_agent(
    llm,
    tools=[search_pdf],
    system_prompt=system_prompt,
    response_format=AgentResponse,
)

company = "Visa"
query = f"Give me the Net Revenue from {company} in billions of dollars for Q1."

# result = agent.invoke({"messages": [HumanMessage(content=query)]})
# for message in reversed(result["messages"]):
#     if message.content:
#         pprint(message)

# pprint(result['messages'][-1].to_json())

# structured_output: Optional[AgentResponse] = result.get("structured_response")

# if structured_output and structured_output.found and structured_output.metrics:
#     pprint(structured_output.metrics)
# else:
#     last_message = result["messages"][-1]
#     if last_message.content:
#         print(f"Model generated raw text response instead of structured response:\n{last_message.content}")
#     else:
#         print(f"Can't find data about {company}")

In [49]:
def build_agent():
    documents: list[Document] = split_pdf_content(pdf_files, max_workers)
    load_embeddings(documents)

    llm = ChatOllama(
        model="qwen2.5:3b",
        temperature=0,
        repeat_penalty=1.18,
        top_k=20,
    )
    system_prompt = """You are a financial data extraction engine.
CRITICAL INSTRUCTION: Your response MUST be ONLY a single valid JSON object.
- NO conversational text (e.g. "Based on the report...")
- NO markdown formatting or code blocks (do NOT use ```json or ```)
- START immediately with { and END with }

EXACT JSON FORMAT TO RETURN:
{
  "found": true,
  "metrics": [
    {
      "company": "Visa",
      "metric_name": "Net Revenue",
      "period": "Q1",
      "value": 10.9,
      "unit": "billion USD"
    }
  ],
  "notes": "15% year-over-year increase"
}

If requested data is not found in the documents:
{
  "found": false,
  "metrics": [],
  "notes": "Information not present in context"
}"""
    return create_agent(
        llm,
        tools=[search_pdf],
        system_prompt=system_prompt,
        response_format=AgentResponse,
    )

In [71]:
import asyncio
import json
import re
import socket
import subprocess
import threading
import time
import traceback
import urllib.request
from contextlib import asynccontextmanager
from typing import Optional, Any, Dict

from fastapi import FastAPI, HTTPException, UploadFile
from fastapi.responses import JSONResponse
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
import nest_asyncio
from pydantic import BaseModel
import uvicorn

nest_asyncio.apply()

# Global state
server_thread: Optional[threading.Thread] = None
server_instance: Optional[uvicorn.Server] = None
vector_store: Optional[Chroma] = None
agent_instance = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    global vector_store, agent_instance

    !ollama pull nomic-embed-text

    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vector_store = Chroma(
        collection_name="financial_docs",
        embedding_function=embeddings,
        persist_directory="./chroma_db",
    )

    agent_instance = build_agent()

    yield


app = FastAPI(lifespan=lifespan)

class Question(BaseModel):
    question: str


@app.get("/")
async def root():
    return {"message": "Hello World"}


@app.post("/upload")
async def upload_report(file: UploadFile | None = None):
    if not file:
        return {"error": "no file uploaded"}

    with open(f"./{file.filename}", "wb") as f:
        data: bytes = await file.read()
        f.write(data)

    return {"file name": file.filename}

@app.post("/ask")
async def ask(question: Question):
    if not question or not question.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")

    try:
        result = await asyncio.wait_for(
            agent_instance.ainvoke(
                {"messages": [HumanMessage(content=question.question)]}
            ),
            timeout=4000,
        )

        structured_output: Optional[AgentResponse] = result.get(
            "structured_response"
        )
        if (
            structured_output
            and structured_output.found
            and structured_output.metrics
        ):
            return structured_output

        last_msg = result["messages"][-1] if result.get("messages") else None
        raw_text = last_msg.content if last_msg else ""

        return {
            "found": False,
            "message": "Model failed to produce conforming structured response.",
            "raw_text": raw_text,
        }

    except asyncio.TimeoutError:
        raise HTTPException(
            status_code=504, detail="Agent query processing timed out"
        )
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={
                "error_type": type(e).__name__,
                "error_message": str(e),
                "traceback": traceback.format_exc().splitlines(),
            },
        )

@app.get("/debug")
async def debug():
    return {
        "ollama": True,
        "chroma": vector_store is not None,
        "agent": agent_instance is not None,
    }


def is_port_in_use(port: int = 8000, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex((host, port)) == 0


def start_colab_server(app, host="127.0.0.1", port=8000):
    global server_thread, server_instance

    if server_instance is not None:
        print("Stopping server")
        server_instance.should_exit = True
        for _ in range(6):
            if not is_port_in_use(port, host):
                break
            time.sleep(0.5)

    if is_port_in_use(port, host):
        print("Port is used")
        time.sleep(2.0)

    def run_server():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        config = uvicorn.Config(
            app, host=host, port=port, log_level="warning"
        )
        global server_instance
        server_instance = uvicorn.Server(config)

        loop.run_until_complete(server_instance.serve())

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()

    start_time = time.time()
    while time.time() - start_time < 15:
        try:
            with urllib.request.urlopen(f"http://{host}:{port}/debug") as resp:
                if resp.status == 200:
                    elapsed = round(time.time() - start_time, 1)
                    print("Fastapi server ready")
                    return True
        except Exception:
            time.sleep(0.5)

    return False


def stop_colab_server():
    global server_instance
    if server_instance is not None:
        server_instance.should_exit = True
        print("Server shutting down")


start_colab_server(app)

Port is used
Fastapi server ready


True

In [56]:
# !fuser -k 8000/tcp

In [70]:
stop_colab_server()

Server shutting down


In [72]:
!curl http://127.0.0.1:8000/

{"message":"Hello World"}

In [73]:
!curl -F "file=@./test_uploads/nvidia_q3.pdf;type=application/pdf" http://127.0.0.1:8000/upload

{"file name":"nvidia_q3.pdf"}

In [74]:
!curl --max-time 4000 -X POST "http://127.0.0.1:8000/ask" \
     -H "Content-Type: application/json" \
     -d '{"question": "Give me the Net Revenue from Visa in billions of dollars for Q1."}'

{"found":true,"metrics":[{"company":"Visa","metric_name":"Net Revenue","period":"Q1","value":10.9,"unit":"billion USD"}],"notes":"15% year-over-year increase"}

In [75]:
!curl --max-time 4000 -X POST "http://127.0.0.1:8000/ask" \
     -H "Content-Type: application/json" \
     -d '{"question": "Give me the Net Revenue from Nvidia in billions of dollars for Q3."}'

{"found":true,"metrics":[{"company":"NVIDIA","metric_name":"Net Revenue","period":"Q3","value":57.0,"unit":"billion USD"}],"notes":""}

In [76]:
!curl http://127.0.0.1:8000/debug

{"ollama":true,"chroma":true,"agent":true}

In [19]:
!#TODO: Upload url string from financial pdf report, then download the pdf to rag as a route